In [ ]:
# %% [markdown]
# # Pilgrim Footfall Prediction System
# 
# This notebook predicts daily visitor footfall at pilgrimage sites using:
# - **Prophet** (baseline time-series model)
# - **LSTM** (deep learning model)
# 
# We'll generate synthetic data, perform EDA, train models, evaluate them, and forecast the next 30 days.

# %% [markdown]
# ## 1. Import Libraries

# %%
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

# Scikit-learn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Prophet
from prophet import Prophet

# TensorFlow/Keras for LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
tf.random.set_seed(42)

print("All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

# %% [markdown]
# ## 2. Generate Synthetic Dataset
# 
# We create 2 years of daily data (730 rows) with realistic patterns:
# - **Seasonal trends** (higher footfall during festival seasons)
# - **Weekly patterns** (weekends typically busier)
# - **Weather effects** (rain reduces footfall)
# - **Festival and holiday spikes**

# %%
def generate_synthetic_data(start_date='2024-01-01', num_days=730):
    """
    Generate synthetic pilgrimage footfall data with realistic patterns.
    """
    dates = pd.date_range(start=start_date, periods=num_days, freq='D')
    
    data = {
        'date': dates,
        'day_of_week': dates.dayofweek,  # 0=Monday, 6=Sunday
        'month': dates.month,
        'day': dates.day,
    }
    df = pd.DataFrame(data)
    
    # Base footfall with seasonal pattern (higher in Oct-Dec and Mar-Apr)
    base_footfall = 5000
    seasonal_pattern = np.sin(2 * np.pi * (df['month'] - 3) / 12) * 2000
    
    # Weekly pattern (weekends are busier)
    weekly_pattern = np.where(df['day_of_week'] >= 5, 1500, 0)
    
    # Temperature (seasonal, 15-40°C range)
    df['temperature'] = 25 + 10 * np.sin(2 * np.pi * (df['month'] - 4) / 12) + np.random.normal(0, 3, num_days)
    df['temperature'] = df['temperature'].clip(15, 42)
    
    # Rainfall (more in monsoon: Jun-Sep)
    monsoon_months = df['month'].isin([6, 7, 8, 9])
    df['rainfall'] = np.where(
        monsoon_months,
        np.random.exponential(15, num_days),
        np.random.exponential(2, num_days)
    )
    df['rainfall'] = df['rainfall'].clip(0, 100)
    
    # Holidays (random ~5% of days)
    df['holiday'] = np.random.choice([0, 1], size=num_days, p=[0.95, 0.05])
    
    # Festivals (specific dates each year - major Hindu festivals)
    festival_dates = []
    for year in df['date'].dt.year.unique():
        # Diwali season (Oct-Nov)
        festival_dates.extend(pd.date_range(f'{year}-10-20', periods=10))
        # Holi (March)
        festival_dates.extend(pd.date_range(f'{year}-03-10', periods=5))
        # Navratri (Sep-Oct)
        festival_dates.extend(pd.date_range(f'{year}-09-25', periods=9))
        # Maha Shivaratri (Feb-Mar)
        festival_dates.extend(pd.date_range(f'{year}-02-25', periods=3))
    
    df['festival'] = df['date'].isin(festival_dates).astype(int)
    
    # Calculate footfall
    df['footfall'] = (
        base_footfall 
        + seasonal_pattern 
        + weekly_pattern
        + df['holiday'] * np.random.uniform(3000, 5000, num_days)  # Holiday boost
        + df['festival'] * np.random.uniform(8000, 15000, num_days)  # Festival spike
        - df['rainfall'] * 50  # Rain reduces footfall
        + np.random.normal(0, 500, num_days)  # Random noise
    )
    
    # Ensure positive footfall
    df['footfall'] = df['footfall'].clip(500, None).astype(int)
    
    return df

# Generate the dataset
df = generate_synthetic_data()

print(f"Dataset shape: {df.shape}")
print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nFirst 10 rows:")
df.head(10)

# %% [markdown]
# ## 3. Data Preprocessing

# %%
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# %%
# Handle any missing values (none expected in synthetic data, but good practice)
df = df.fillna(method='ffill').fillna(method='bfill')

# Ensure date is datetime
df['date'] = pd.to_datetime(df['date'])

# Extract additional time features
df['weekday'] = df['day_of_week']  # Already have this
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['quarter'] = df['date'].dt.quarter
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)

print("Feature engineering complete!")
print(f"\nUpdated columns: {df.columns.tolist()}")
print(f"\nDataset statistics:")
df.describe()

# %% [markdown]
# ## 4. Exploratory Data Analysis (EDA)

# %% [markdown]
# ### 4.1 Footfall Over Time

# %%
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Full time series
axes[0].plot(df['date'], df['footfall'], color='steelblue', linewidth=0.8)
axes[0].set_title('Daily Pilgrim Footfall Over Time', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Footfall')
axes[0].axhline(y=df['footfall'].mean(), color='red', linestyle='--', label=f"Mean: {df['footfall'].mean():.0f}")
axes[0].legend()

# Monthly average
monthly_avg = df.groupby(df['date'].dt.to_period('M'))['footfall'].mean()
monthly_avg.index = monthly_avg.index.to_timestamp()
axes[1].bar(monthly_avg.index, monthly_avg.values, width=20, color='coral', edgecolor='darkred')
axes[1].set_title('Monthly Average Footfall', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Footfall')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# %% [markdown]
# ### 4.2 Correlation Heatmap

# %%
# Select numeric columns for correlation
numeric_cols = ['footfall', 'temperature', 'rainfall', 'holiday', 'festival', 
                'day_of_week', 'month', 'is_weekend']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlBu_r', 
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey correlations with footfall:")
print(corr_matrix['footfall'].sort_values(ascending=False))

# %% [markdown]
# ### 4.3 Identify Peak Days

# %%
# Find top 20 peak days
peak_days = df.nlargest(20, 'footfall')[['date', 'footfall', 'festival', 'holiday', 'day_of_week']]
peak_days['day_name'] = peak_days['date'].dt.day_name()

print("Top 20 Peak Footfall Days:")
print(peak_days.to_string(index=False))

# Visualize peak days
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(peak_days)), peak_days['footfall'], color='crimson')
ax.set_xticks(range(len(peak_days)))
ax.set_xticklabels(peak_days['date'].dt.strftime('%Y-%m-%d'), rotation=45, ha='right')
ax.set_title('Top 20 Peak Footfall Days', fontsize=14, fontweight='bold')
ax.set_ylabel('Footfall')
plt.tight_layout()
plt.show()

# %% [markdown]
# ### 4.4 Footfall by Day of Week and Festival Impact

# %%
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By day of week
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
day_avg = df.groupby('day_of_week')['footfall'].mean()
axes[0].bar(day_names, day_avg.values, color='teal', edgecolor='darkslategray')
axes[0].set_title('Average Footfall by Day of Week', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average Footfall')

# Festival vs non-festival
festival_comparison = df.groupby('festival')['footfall'].mean()
axes[1].bar(['Non-Festival', 'Festival'], festival_comparison.values, 
            color=['lightgray', 'gold'], edgecolor='black')
axes[1].set_title('Festival vs Non-Festival Footfall', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Average Footfall')

plt.tight_layout()
plt.show()

# %% [markdown]
# ## 5. Prepare Data for Modeling

# %%
# Train-test split (use last 60 days as test set)
train_size = len(df) - 60
train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

print(f"Training set: {len(train_df)} days ({train_df['date'].min()} to {train_df['date'].max()})")
print(f"Test set: {len(test_df)} days ({test_df['date'].min()} to {test_df['date'].max()})")

# %% [markdown]
# ## 6. Model 1: Prophet (Baseline Time-Series Model)

# %%
# Prepare data for Prophet (requires 'ds' and 'y' columns)
prophet_train = train_df[['date', 'footfall']].copy()
prophet_train.columns = ['ds', 'y']

prophet_test = test_df[['date', 'footfall']].copy()
prophet_test.columns = ['ds', 'y']

# Add regressors
for col in ['temperature', 'rainfall', 'holiday', 'festival']:
    prophet_train[col] = train_df[col].values
    prophet_test[col] = test_df[col].values

print("Prophet data prepared!")
prophet_train.head()

# %%
# Initialize and train Prophet model
prophet_model = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.05,
    seasonality_mode='multiplicative'
)

# Add custom regressors
prophet_model.add_regressor('temperature')
prophet_model.add_regressor('rainfall')
prophet_model.add_regressor('holiday')
prophet_model.add_regressor('festival')

# Fit the model
prophet_model.fit(prophet_train)
print("Prophet model trained successfully!")

# %%
# Predict on test set
prophet_predictions = prophet_model.predict(prophet_test[['ds', 'temperature', 'rainfall', 'holiday', 'festival']])

# Extract predictions
prophet_pred_values = prophet_predictions['yhat'].values
prophet_actual = prophet_test['y'].values

# Calculate metrics
prophet_mae = mean_absolute_error(prophet_actual, prophet_pred_values)
prophet_rmse = np.sqrt(mean_squared_error(prophet_actual, prophet_pred_values))

print("\n" + "="*50)
print("PROPHET MODEL EVALUATION")
print("="*50)
print(f"MAE:  {prophet_mae:.2f}")
print(f"RMSE: {prophet_rmse:.2f}")
print("="*50)

# %% [markdown]
# ## 7. Model 2: LSTM (Deep Learning Model)

# %%
# Prepare data for LSTM
features = ['footfall', 'temperature', 'rainfall', 'holiday', 'festival', 
            'day_of_week', 'month', 'is_weekend']

# Scale the data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[features])

# Create a separate scaler for footfall (for inverse transform)
footfall_scaler = MinMaxScaler()
footfall_scaler.fit(df[['footfall']])

print(f"Scaled data shape: {scaled_data.shape}")

# %%
def create_sequences(data, seq_length):
    """
    Create sequences for LSTM training.
    Each sequence contains seq_length days of data to predict the next day's footfall.
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length, 0])  # Footfall is the first column
    return np.array(X), np.array(y)

# Sequence length (use 30 days of history)
SEQ_LENGTH = 30

# Create sequences
X, y = create_sequences(scaled_data, SEQ_LENGTH)

# Split into train and test
train_size_lstm = len(train_df) - SEQ_LENGTH
X_train, X_test = X[:train_size_lstm], X[train_size_lstm:]
y_train, y_test = y[:train_size_lstm], y[train_size_lstm:]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# %%
# Build LSTM model
lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LENGTH, len(features))),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print(lstm_model.summary())

# %%
# Train LSTM model
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = lstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

print("\nLSTM model trained successfully!")

# %%
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('LSTM Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()

axes[1].plot(history.history['mae'], label='Training MAE')
axes[1].plot(history.history['val_mae'], label='Validation MAE')
axes[1].set_title('LSTM Training MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.show()

# %%
# Predict on test set
lstm_pred_scaled = lstm_model.predict(X_test, verbose=0)

# Inverse transform predictions
lstm_pred_values = footfall_scaler.inverse_transform(lstm_pred_scaled).flatten()
lstm_actual = footfall_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Calculate metrics
lstm_mae = mean_absolute_error(lstm_actual, lstm_pred_values)
lstm_rmse = np.sqrt(mean_squared_error(lstm_actual, lstm_pred_values))

print("\n" + "="*50)
print("LSTM MODEL EVALUATION")
print("="*50)
print(f"MAE:  {lstm_mae:.2f}")
print(f"RMSE: {lstm_rmse:.2f}")
print("="*50)

# %% [markdown]
# ## 8. Model Comparison

# %%
# Summary comparison
comparison_df = pd.DataFrame({
    'Model': ['Prophet', 'LSTM'],
    'MAE': [prophet_mae, lstm_mae],
    'RMSE': [prophet_rmse, lstm_rmse]
})

print("\n" + "="*50)
print("MODEL COMPARISON SUMMARY")
print("="*50)
print(comparison_df.to_string(index=False))
print("="*50)

# Determine best model
best_model = 'Prophet' if prophet_rmse < lstm_rmse else 'LSTM'
print(f"\n✓ Best performing model based on RMSE: {best_model}")

# %% [markdown]
# ## 9. Visualize Actual vs Predicted

# %%
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Prophet predictions
test_dates = test_df['date'].values[:len(prophet_pred_values)]
axes[0].plot(test_dates, prophet_actual, label='Actual', color='blue', linewidth=2)
axes[0].plot(test_dates, prophet_pred_values, label='Prophet Predicted', color='orange', linewidth=2, linestyle='--')
axes[0].fill_between(test_dates, 
                     prophet_predictions['yhat_lower'].values, 
                     prophet_predictions['yhat_upper'].values, 
                     alpha=0.2, color='orange', label='Confidence Interval')
axes[0].set_title('Prophet: Actual vs Predicted Footfall', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Footfall')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# LSTM predictions (adjust for sequence length offset)
lstm_test_dates = test_df['date'].values[:len(lstm_pred_values)]
axes[1].plot(lstm_test_dates, lstm_actual, label='Actual', color='blue', linewidth=2)
axes[1].plot(lstm_test_dates, lstm_pred_values, label='LSTM Predicted', color='green', linewidth=2, linestyle='--')
axes[1].set_title('LSTM: Actual vs Predicted Footfall', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Footfall')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# %% [markdown]
# ## 10. Forecast Next 30 Days

# %%
# Generate future dates
last_date = df['date'].max()
future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30, freq='D')

# Create future dataframe with estimated features
future_df = pd.DataFrame({'date': future_dates})
future_df['ds'] = future_df['date']
future_df['day_of_week'] = future_df['date'].dt.dayofweek
future_df['month'] = future_df['date'].dt.month
future_df['day'] = future_df['date'].dt.day
future_df['is_weekend'] = (future_df['day_of_week'] >= 5).astype(int)

# Estimate weather (use historical averages for the same period)
future_df['temperature'] = 30 + 5 * np.sin(2 * np.pi * (future_df['month'] - 4) / 12)
future_df['rainfall'] = np.where(future_df['month'].isin([6, 7, 8, 9]), 10, 2)

# Assume no special holidays/festivals (can be customized)
future_df['holiday'] = 0
future_df['festival'] = 0

print("Future data prepared for forecasting:")
future_df.head(10)

# %%
# Prophet forecast
prophet_future = future_df[['ds', 'temperature', 'rainfall', 'holiday', 'festival']].copy()
prophet_forecast = prophet_model.predict(prophet_future)

future_df['prophet_forecast'] = prophet_forecast['yhat'].values
future_df['prophet_lower'] = prophet_forecast['yhat_lower'].values
future_df['prophet_upper'] = prophet_forecast['yhat_upper'].values

print("Prophet 30-day forecast complete!")

# %%
# LSTM forecast (rolling prediction)
# Use the last SEQ_LENGTH days as initial input
last_sequence = scaled_data[-SEQ_LENGTH:].copy()

lstm_future_predictions = []

for i in range(30):
    # Predict next day
    pred = lstm_model.predict(last_sequence.reshape(1, SEQ_LENGTH, len(features)), verbose=0)
    lstm_future_predictions.append(pred[0, 0])
    
    # Create next day's features (scaled)
    next_day_features = np.zeros(len(features))
    next_day_features[0] = pred[0, 0]  # Predicted footfall
    
    # Scale the known features for the future day
    future_row = future_df.iloc[i]
    next_day_features[1] = (future_row['temperature'] - df['temperature'].min()) / (df['temperature'].max() - df['temperature'].min())
    next_day_features[2] = (future_row['rainfall'] - df['rainfall'].min()) / (df['rainfall'].max() - df['rainfall'].min())
    next_day_features[3] = future_row['holiday']
    next_day_features[4] = future_row['festival']
    next_day_features[5] = future_row['day_of_week'] / 6
    next_day_features[6] = (future_row['month'] - 1) / 11
    next_day_features[7] = future_row['is_weekend']
    
    # Roll the sequence
    last_sequence = np.vstack([last_sequence[1:], next_day_features])

# Inverse transform LSTM predictions
lstm_future_scaled = np.array(lstm_future_predictions).reshape(-1, 1)
future_df['lstm_forecast'] = footfall_scaler.inverse_transform(lstm_future_scaled).flatten()

print("LSTM 30-day forecast complete!")

# %% [markdown]
# ## 11. Display Forecast Results

# %%
# Calculate average forecast and identify peak days
future_df['avg_forecast'] = (future_df['prophet_forecast'] + future_df['lstm_forecast']) / 2
future_df['day_name'] = future_df['date'].dt.day_name()

# Identify peak days (top 25% of forecasted footfall)
peak_threshold = future_df['avg_forecast'].quantile(0.75)
future_df['is_peak'] = future_df['avg_forecast'] >= peak_threshold

# Display forecast table
print("\n" + "="*80)
print("30-DAY FOOTFALL FORECAST")
print("="*80)

display_df = future_df[['date', 'day_name', 'prophet_forecast', 'lstm_forecast', 'avg_forecast', 'is_peak']].copy()
display_df['prophet_forecast'] = display_df['prophet_forecast'].astype(int)
display_df['lstm_forecast'] = display_df['lstm_forecast'].astype(int)
display_df['avg_forecast'] = display_df['avg_forecast'].astype(int)
display_df['is_peak'] = display_df['is_peak'].map({True: '⚠️ PEAK', False: ''})
display_df.columns = ['Date', 'Day', 'Prophet', 'LSTM', 'Average', 'Alert']

print(display_df.to_string(index=False))
print("="*80)

# %%
# Highlight peak days
peak_days_forecast = future_df[future_df['is_peak']][['date', 'day_name', 'avg_forecast']]
peak_days_forecast.columns = ['Date', 'Day', 'Expected Footfall']
peak_days_forecast['Expected Footfall'] = peak_days_forecast['Expected Footfall'].astype(int)

print("\n⚠️  PEAK DAY ALERTS (High Expected Footfall):")
print("-" * 50)
print(peak_days_forecast.to_string(index=False))

# %% [markdown]
# ## 12. Visualize Future Forecast

# %%
fig, ax = plt.subplots(figsize=(14, 6))

# Historical data (last 60 days)
historical = df.tail(60)

# Plot historical
ax.plot(historical['date'], historical['footfall'], 
        label='Historical', color='blue', linewidth=2)

# Plot Prophet forecast
ax.plot(future_df['date'], future_df['prophet_forecast'], 
        label='Prophet Forecast', color='orange', linewidth=2, linestyle='--')

# Plot LSTM forecast
ax.plot(future_df['date'], future_df['lstm_forecast'], 
        label='LSTM Forecast', color='green', linewidth=2, linestyle=':')

# Plot average forecast
ax.plot(future_df['date'], future_df['avg_forecast'], 
        label='Average Forecast', color='red', linewidth=2.5)

# Confidence interval for Prophet
ax.fill_between(future_df['date'], 
                future_df['prophet_lower'], 
                future_df['prophet_upper'], 
                alpha=0.15, color='orange')

# Mark peak days
peak_dates = future_df[future_df['is_peak']]['date']
peak_values = future_df[future_df['is_peak']]['avg_forecast']
ax.scatter(peak_dates, peak_values, color='red', s=100, zorder=5, 
           marker='*', label='Peak Days')

# Add vertical line separating historical and forecast
ax.axvline(x=last_date, color='gray', linestyle='--', alpha=0.7, label='Forecast Start')

ax.set_title('Pilgrim Footfall: Historical Data & 30-Day Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Footfall')
ax.legend(loc='upper left')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# %% [markdown]
# ## 13. Save Models and Data for Streamlit App

# %%
# Save the trained models and data
import pickle

# Save Prophet model
with open('prophet_model.pkl', 'wb') as f:
    pickle.dump(prophet_model, f)

# Save LSTM model
lstm_model.save('lstm_model.h5')

# Save scalers
with open('scalers.pkl', 'wb') as f:
    pickle.dump({
        'scaler': scaler,
        'footfall_scaler': footfall_scaler
    }, f)

# Save historical data
df.to_csv('historical_data.csv', index=False)

# Save feature list
with open('features.pkl', 'wb') as f:
    pickle.dump(features, f)

print("✓ All models and data saved successfully!")
print("\nFiles created:")
print("  - prophet_model.pkl")
print("  - lstm_model.h5")
print("  - scalers.pkl")
print("  - historical_data.csv")
print("  - features.pkl")

# %% [markdown]
# ## Summary
# 
# This notebook successfully:
# 
# 1. **Generated synthetic data** with 730 days of realistic pilgrimage footfall patterns
# 2. **Performed EDA** to understand trends, correlations, and peak days
# 3. **Trained two models**:
#    - Prophet (time-series baseline)
#    - LSTM (deep learning)
# 4. **Evaluated both models** using MAE and RMSE metrics
# 5. **Forecasted the next 30 days** with both models
# 6. **Identified peak days** that require attention
# 7. **Saved models** for use in the Streamlit dashboard
# 
# The Streamlit app (`app.py`) can now load these models and provide an interactive forecasting dashboard.
